# ASM2d-TCN Simulation

This notebook runs the ASM2d-TCN simulation model from the source package and persists the generated dataset and metadata under `data/asm2d-tcn/simulation`.

Unlike the ASM1 orchestration notebook, the reported output variables here are exclusively the configured composites.

In [ ]:
import pandas as pd
from src.models.simulation.asm2d_tcn_simulation import (
    run_asm2d_tcn_simulation,
    sweep_asm2d_tcn_operating_space,
 )

simulation_result = run_asm2d_tcn_simulation(
    save_artifacts=True,
    include_debug_data=True,
    show_progress=True,
    progress_description="ASM2d-TCN dataset",
)

dataset = simulation_result["dataset"]
metadata = simulation_result["metadata"]
artifact_paths = simulation_result["artifact_paths"]
petersen_matrix = simulation_result["petersen_matrix"]
composition_matrix = simulation_result["composition_matrix"]
effluent_states = simulation_result["effluent_states"]
solver_diagnostics = simulation_result["solver_diagnostics"]
solver_summary = simulation_result["solver_summary"]

print(f"Generated {len(dataset)} rows for {metadata['simulation_name']}.")
print(f"Dataset saved to: {artifact_paths['dataset_csv']}")
print(f"Metadata saved to: {artifact_paths['metadata_json']}")
print(f"Petersen matrix shape: {petersen_matrix.shape}")
print(f"Composition matrix shape: {composition_matrix.shape}")
print("Saved dataset contract remains composite-only; debug payloads stay in memory.")

output_columns = [column_name for column_name in metadata["dependent_columns"] if column_name.startswith("Out_")]
print(f"Reported output variables: {', '.join(output_columns)}")

display(dataset.head())
display(pd.Series(metadata, name="value").to_frame())
display(pd.json_normalize(solver_summary, sep="."))
display(solver_diagnostics.head())
display(effluent_states.head())
display(pd.DataFrame(petersen_matrix, index=metadata["processes"], columns=metadata["state_columns"]))
display(pd.DataFrame(composition_matrix, index=metadata["measured_output_columns"], columns=metadata["state_columns"]))

c:\Users\eselerio\projects\pibre-model\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
ASM2d-TCN dataset:   0%|          | 0/10000 [00:00<?, ?sample/s]

In [2]:
import numpy as np
from scipy.linalg import null_space

cobre_constraint_basis = null_space(petersen_matrix)
cobre_A_matrix = cobre_constraint_basis.T

cobre_A_matrix = np.round(cobre_A_matrix, 5)
cobre_A_matrix[np.abs(cobre_A_matrix) < 1e-10] = 0.0

for row_index in range(cobre_A_matrix.shape[0]):
    non_zero_entries = cobre_A_matrix[row_index, cobre_A_matrix[row_index, :] != 0]
    if len(non_zero_entries) > 0:
        cobre_A_matrix[row_index, :] = cobre_A_matrix[row_index, :] / non_zero_entries[0]

macroscopic_stoichiometric_matrix = petersen_matrix @ composition_matrix.T
measured_constraint_basis = null_space(macroscopic_stoichiometric_matrix)
A_matrix = measured_constraint_basis.T

A_matrix = np.round(A_matrix, 5)
A_matrix[np.abs(A_matrix) < 1e-10] = 0.0

for row_index in range(A_matrix.shape[0]):
    non_zero_entries = A_matrix[row_index, A_matrix[row_index, :] != 0]
    if len(non_zero_entries) > 0:
        A_matrix[row_index, :] = A_matrix[row_index, :] / non_zero_entries[0]

print(f"Fractional Petersen matrix shape: {petersen_matrix.shape}")
print(f"COBRE invariant matrix shape: {cobre_A_matrix.shape}")
print(f"Measured-space invariant matrix shape kept for downstream regressors: {A_matrix.shape}")
display(
    pd.DataFrame(
        cobre_A_matrix,
        index=[f"constraint_{index + 1}" for index in range(cobre_A_matrix.shape[0])],
        columns=metadata["state_columns"],
    )
)
display(
    pd.DataFrame(
        A_matrix,
        index=[f"constraint_{index + 1}" for index in range(A_matrix.shape[0])],
        columns=metadata["measured_output_columns"],
    )
)

Fractional Petersen matrix shape: (28, 21)
COBRE invariant matrix shape: (6, 21)
Measured-space invariant matrix shape kept for downstream regressors: (0, 6)


,S_A,S_F,S_I,S_N2,S_NH4,S_NO2,S_NO3,S_PO4,S_ALK,S_O2,...,X_S,X_H,X_PAO,X_PP,X_PHA,X_AOB,X_NOB,X_TSS,X_MeOH,X_MeP
constraint_1,-0.0,1.0,-122.115741,43.285494,43.180556,43.391975,43.391975,-29.893519,1.476852,-0.0,...,6.891975,8.584877,8.584877,-6.334877,4.367284,8.584877,8.584877,-7.279321,2.435185,-2.280864
constraint_2,0.0,1.0,41.259080,30.538741,30.934625,30.142857,30.142857,8.578692,-5.548426,0.0,...,-1.389831,0.099274,0.099274,-3.207022,-2.156174,0.099274,0.099274,3.593220,77.556901,55.619855
constraint_3,0.0,1.0,-17.963673,7.606458,9.781029,5.431887,5.431887,78.202825,-30.441978,0.0,...,-17.095863,-15.973764,-15.973764,-1.039354,-14.537841,-15.973764,-15.973764,24.229062,-13.890010,-1.048436
constraint_4,0.0,1.0,50.768730,34.358306,33.546145,35.171553,35.171553,-3.445168,11.373507,0.0,...,1.364821,2.915309,2.915309,-2.984799,0.017372,2.915309,2.915309,-0.029316,-52.864278,-38.073833
constraint_5,-0.0,1.0,0.197026,7.572491,11.695167,3.449814,3.449814,79.107807,-57.717472,-0.0,...,56.650558,57.769517,57.769517,316.583643,44.460967,57.769517,57.769517,-74.100372,-20.063197,23.252788
constraint_6,-0.0,1.0,-31.996269,-6.250000,-30.932836,18.432836,18.432836,107.679104,345.544776,-0.0,...,-13.630597,-12.731343,-12.731343,56.078358,-11.656716,-12.731343,-12.731343,19.425373,11.552239,26.917910


,COD,TN,TKN,TP,TSS,VSS


## Solver Sweep

This section samples the configured operating space without changing the saved dataset contract. It summarizes how often the current acceptance threshold is met, how often the multistart guess wins, and how often dynamic relaxation is needed.

In [ ]:
SWEEP_SAMPLE_COUNT = 1024

sweep_result = sweep_asm2d_tcn_operating_space(
    n_samples=SWEEP_SAMPLE_COUNT,
    random_seed=int(metadata["random_seed"]),
    show_progress=True,
    progress_description="ASM2d-TCN operating-space sweep",
)

sweep_summary = sweep_result["summary"]
sweep_diagnostics = sweep_result["solver_diagnostics"].copy()
sweep_diagnostics["threshold_margin"] = (
    sweep_diagnostics["acceptance_threshold"] - sweep_diagnostics["residual_max"]
)

print(f"Operating-space sweep completed for {sweep_summary['sample_count']} sampled points.")
print(
    f"Accepted points at threshold {sweep_summary['acceptance_threshold']}: "
    f"{sweep_summary['accepted_count']} ({sweep_summary['accepted_rate']:.1%})"
 )
print(f"Multistart selected rate: {sweep_summary['multistart_selected_rate']:.1%}")
print(f"Dynamic relaxation used rate: {sweep_summary['dynamic_relaxation_used_rate']:.1%}")
print(f"Dynamic relaxation improved rate: {sweep_summary['dynamic_relaxation_improved_rate']:.1%}")

display(pd.json_normalize(sweep_summary, sep="."))
display(sweep_diagnostics.sort_values("residual_max", ascending=False).head(20))